# NIVAL airborne lidar versus NISAR dSWE, using SnowIn

This notebook adapts the NISAR comparison in Zach Hoppinen's [tc_brief_nival_validation](https://github.com/ZachHoppinen/tc_brief_nival_validation) repository to SnowIn. It targets Mores Creek Summit, Idaho, ASC-077, 7–19 February 2026. The reference analysis uses the operational NISAR L2 GUNW at 80 m and NIVAL lidar snow-depth rasters from 7 and 22 February. The processing output can be written in the reference repository's existing NetCDF layout so its Figure 2 plotting function can be reused unchanged.

The workflow removes the scene median phase before conversion; computes dSWE with SnowIn's canonical kernel, product wavelength and local incidence angle; differences the two lidar surfaces on their native grid; masks |dHS| ≥ 1.5 m there; average-resamples the gated change to the GUNW grid; anchors dSWE to median lidar-implied SWE at 147 kg/m³; and reports correlation, RMSE, fitted density and sample count by coherence gate. The phase datum removal happens before the angle-dependent conversion. The absolute SWE anchor is a separate constant offset that does not affect correlation. Zach's current code uses fixed wavelength 0.238 m; this notebook uses the GUNW metadata wavelength.

The [EGUsphere preprint](https://doi.org/10.5194/egusphere-2026-5140) reports all-pixel n=4,976, r=0.82 and centered RMSD=19 mm; above coherence 0.2 it reports r=0.93 and 12 mm, and above 0.4 r=0.96 and 9 mm. It describes the fitted density as 141 kg/m³ for all pixels and about 154–160 kg/m³ across the 0.2/0.4 gates. The cloned code repository's README instead lists densities 150 / 157 / 161 / 162 kg/m³ for all / >0.2 / >0.4 / >0.5. That code-versus-preprint difference should be resolved by running the checked-out analysis; this notebook calculates the slope and does not force either set. Zach's detailed stats script also includes >0.3, elevation deramping, scale decomposition, canopy/burn strata and local correlations, which need more inputs and are outside this minimal port.

The inputs can be downloaded with `python -m pip install -e '.[gunw]'` followed by `python scripts/download_nival_nisar_inputs.py --inputs lidar --interactive-login` and `python scripts/download_nival_nisar_inputs.py --inputs gunw --interactive-login`. This fetches just the two snow-depth GeoTIFF assets from the free [NSIDC NIVAL collection](https://doi.org/10.5067/DPFDH2M49DQG) plus the [ASF GUNW](https://doi.org/10.5067/NIL2GUNW-P1), about 2.5 GB. A free NASA Earthdata Login is required; the interactive option bypasses a stale local `.netrc`. Data go under `~/.cache/snowin/nival_nisar/` by default.

The correction screens carried in the GUNW are not applied, matching Zach's stated choice. Numerical differences can result from wavelength, LIA derivation, resampling, or valid-data masks.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import rioxarray
from pyproj import Transformer
import xarray as xr
from rasterio.enums import Resampling
from snowin import compute_dswe
from snowin.io import add_gunw_incidence, open_gunw

DATA_DIR = Path.home() / '.cache' / 'snowin' / 'nival_nisar'
GUNW = DATA_DIR / 'gunw' / 'NISAR_L2_PR_GUNW_012_077_A_024_013_4000_SH_20260207T124619_20260207T124654_20260219T124619_20260219T124654_P05023_N_F_J_001.h5'
LIDAR_FEB07 = DATA_DIR / 'lidar' / 'NIVAL_MCS_Lidar_20260207_SD_v01.tif'
LIDAR_FEB22 = DATA_DIR / 'lidar' / 'NIVAL_MCS_Lidar_20260222_SD_v01.tif'
SNOW_DENSITY_KG_M3, WATER_DENSITY_KG_M3 = 147.0, 1000.0
DHS_GATE_M = 1.5
for path in (GUNW, LIDAR_FEB07, LIDAR_FEB22):
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

In [ ]:
# Open the operational full-frame product, then crop to Zach's Mores Creek AOI.
pair = open_gunw(GUNW, chunks=None, progress=True)
aoi = (-115.745, 43.898, -115.620, 43.993)  # west, south, east, north, EPSG:4326
epsg = int(pair['spatial_ref'].attrs['epsg_code'])
to_projected = Transformer.from_crs(4326, epsg, always_xy=True).transform
west, south, east, north = aoi
projected = [to_projected(lon, lat) for lon in (west, east) for lat in (south, north)]
x_bounds = (min(point[0] for point in projected), max(point[0] for point in projected))
y_bounds = (min(point[1] for point in projected), max(point[1] for point in projected))
x_index = np.flatnonzero((pair.x.values >= x_bounds[0]) & (pair.x.values <= x_bounds[1]))
y_index = np.flatnonzero((pair.y.values >= y_bounds[0]) & (pair.y.values <= y_bounds[1]))
if not x_index.size or not y_index.size:
    raise ValueError('Mores AOI does not intersect the GUNW grid')
pair = pair.isel(x=x_index, y=y_index)
# Use GLO-30 geometry, like the reference workflow. SnowIn obtains/caches the DEM if needed.
pair = add_gunw_incidence(pair, GUNW, dem_source='cop30', cop30_dem='auto', progress=True)
phase, coherence, lia = pair['phase'], pair['coherence'], pair['incidence_angle']
print({'reference_time': pair.attrs.get('reference_time'), 'secondary_time': pair.attrs.get('secondary_time'), 'wavelength_m': pair.attrs.get('wavelength_m'), 'phase_convention': phase.attrs.get('phase_difference_definition'), 'lia_source': lia.attrs.get('incidence_angle_source'), 'shape': phase.shape, 'posting_m': float(abs(phase.x.values[1] - phase.x.values[0]))})

In [ ]:
# SnowIn's incidence adapter has already supplied local angle in radians.
phase_zeroed = (phase - phase.median(skipna=True)).rename('phase')
phase_zeroed.attrs = dict(phase.attrs)
phase_zeroed.attrs.update({'units': 'rad', 'phase_difference_definition': 'secondary_minus_reference'})
dswe_unanchored = compute_dswe(phase_zeroed, lia, wavelength_m=float(pair.attrs['wavelength_m']))
print('finite phase/dSWE pixels:', int(np.isfinite(dswe_unanchored.values).sum()))

In [ ]:
# Difference and gate dHS on the native lidar grid before average-resampling to 80 m,
# matching Zach's treatment of void/canopy outliers.
spatial_ref = pair['spatial_ref'].attrs
crs = spatial_ref.get('spatial_ref') or spatial_ref.get('crs_wkt')
if crs is None:
    raise ValueError('GUNW spatial_ref has no CRS text')
grid = xr.DataArray(np.zeros(phase.shape, dtype=np.float32), dims=phase.dims, coords={'y': phase.y, 'x': phase.x}, name='grid').rio.write_crs(crs)
depth_earlier = rioxarray.open_rasterio(LIDAR_FEB07, masked=True).squeeze()
depth_later = rioxarray.open_rasterio(LIDAR_FEB22, masked=True).squeeze()
if depth_earlier.shape != depth_later.shape or depth_earlier.rio.crs != depth_later.rio.crs or depth_earlier.rio.transform() != depth_later.rio.transform():
    raise ValueError('The two native lidar rasters must share a grid before differencing')
native_dhs = depth_later - depth_earlier
native_dhs = native_dhs.where((native_dhs > -DHS_GATE_M) & (native_dhs < DHS_GATE_M)).rio.write_nodata(np.nan)
dhs = native_dhs.rio.reproject_match(grid, resampling=Resampling.average).rename('dhs').assign_coords(y=phase.y, x=phase.x)
dhs.attrs.update({'units': 'm', 'interval': '2026-02-07 to 2026-02-22'})
common = np.isfinite(dswe_unanchored.values) & np.isfinite(dhs.values)
if not common.any():
    raise ValueError('No overlapping valid NISAR dSWE and lidar dHS pixels; check CRS, AOI, and nodata')
lidar_swe = dhs.values * SNOW_DENSITY_KG_M3 / WATER_DENSITY_KG_M3
anchor_m = np.nanmedian(lidar_swe[common] - dswe_unanchored.values[common])
dswe = (dswe_unanchored + anchor_m).rename('dswe')
print(f'common pixels: {int(common.sum()):,}; dSWE constant anchor: {anchor_m * 1000:.2f} mm')

In [ ]:
def comparison_metrics(mask=None):
    valid = common.copy()
    if mask is not None:
        valid &= np.asarray(mask, dtype=bool)
    n = int(valid.sum())
    if n < 3:
        return {'n': n, 'r': np.nan, 'centered_rmsd_mm': np.nan, 'density_kg_m3': np.nan}
    x, h = dswe.values[valid], dhs.values[valid]
    lidar = h * SNOW_DENSITY_KG_M3 / WATER_DENSITY_KG_M3
    centered_offset = np.median(lidar - x)
    centered_rmsd_mm = float(np.sqrt(np.mean((x + centered_offset - lidar) ** 2)) * 1000)
    return {'n': n, 'r': float(np.corrcoef(x, h)[0, 1]), 'centered_rmsd_mm': centered_rmsd_mm, 'density_kg_m3': float(np.polyfit(h, x, 1)[0] * WATER_DENSITY_KG_M3)}
gates = [('all pixels', None)] + [(f'coherence > {t:.1f}', coherence.values > t) for t in (0.2, 0.3, 0.4, 0.5)]
rows = []
for label, gate in gates:
    row = comparison_metrics(gate)
    row.update({'gate': label, 'percent_of_valid': 100 * row['n'] / int(common.sum())})
    rows.append(row)
pd.DataFrame(rows)[['gate', 'r', 'centered_rmsd_mm', 'density_kg_m3', 'n', 'percent_of_valid']]

## Save SnowIn output for Zach's plotting code

The operational 80 m layer is already unwrapped in the GUNW. The cell below saves SnowIn's result with the same variable names and coordinates expected by `nival.retrieval.load_product()`. Set `NIVAL_REFERENCE_REPO` to a checkout of Zach's repository before launching Jupyter to also write its expected `outputs/retrieval/dswe_dhs_operational.nc`. Then call `results_figure()` from Zach's existing `nival.paper_figures` module in that repository's `nival` environment; it will create `figures/fig2_results.png`.

Figure 3 follows a different path. It uses the GUNW's 20 m **wrapped** interferogram, which Zach unwraps with SNAPHU. His helper uses ISCE3 to estimate effective looks from the raw RSLC metadata; it does not rebuild the interferogram from RSLCs. SnowIn currently reads the delivered 80 m unwrapped product and does not include a phase-unwrapping engine, so retain Zach's `unwrap_20m.py` for that 20 m diagnostic.

In [ ]:
import os

# Persist a SnowIn-native 80 m result; optionally stage a copy at Zach's expected path.
dem_path = Path(pair.attrs['dem_input'])
elevation = (rioxarray.open_rasterio(dem_path, masked=True).squeeze()
             .rio.reproject_match(grid, resampling=Resampling.bilinear)
             .rename('elevation'))
lia_degrees = (np.rad2deg(lia).rename('lia'))
lia_degrees.attrs.update({'units': 'degrees', 'incidence_angle_reference': 'local'})
snowin_product = xr.Dataset(
    {'dswe': dswe, 'dhs': dhs, 'coherence': coherence, 'elevation': elevation, 'lia': lia_degrees},
    attrs={'crs_epsg': epsg, 'posting_m': float(abs(phase.x.values[1] - phase.x.values[0])),
           'processing': 'SnowIn open_gunw, add_gunw_incidence, and compute_dswe',
           'wavelength_m': float(pair.attrs['wavelength_m']),
           'lidar_snow_density_kg_m3': SNOW_DENSITY_KG_M3})
snowin_output = DATA_DIR / 'snowin_nival_comparison_80m.nc'
snowin_product.to_netcdf(snowin_output)
print('SnowIn product:', snowin_output)

reference_repo = os.environ.get('NIVAL_REFERENCE_REPO')
if reference_repo:
    reference_repo = Path(reference_repo).expanduser().resolve()
    if not (reference_repo / 'src/nival/paper_figures.py').is_file():
        raise FileNotFoundError(f'Not a tc_brief_nival_validation checkout: {reference_repo}')
    compat_output = reference_repo / 'outputs/retrieval/dswe_dhs_operational.nc'
    compat_output.parent.mkdir(parents=True, exist_ok=True)
    snowin_product.to_netcdf(compat_output)
    print('Zach-compatible product:', compat_output)
    print('Render his unchanged Figure 2 function with:')
    print(f"conda run -n nival python -c \"import sys; sys.path.insert(0, '{reference_repo / 'src'}'); from nival.paper_figures import results_figure; results_figure()\"")
else:
    print('Set NIVAL_REFERENCE_REPO to also export the Zach-compatible product for his Figure 2 plotting code.')

## Interpretation and availability

The 147 kg/m³ anchor sets the absolute dSWE level, but centered RMSD is calculated after removing the median dSWE-minus-lidar difference separately within each coherence subset, matching the paper's statistic. Fitted density is the regression slope and is independent of an additive reference offset. Confirm lidar nodata semantics and valid-pixel count before interpreting numerical differences. Zach states that correction screens carried in the GUNW are not applied; this notebook follows that choice.

Analysis code and pair census are in the linked GitHub repository. NIVAL lidar is distributed by NSIDC and the GUNW by ASF at the DOI links above. SNOTEL is available from NRCS. Freeman station measurements are available from the authors on request. The ancillary figures and other datasets are outside this NISAR-only port.